In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sys

REPO_ROOT = Path("..").resolve()
sys.path.append(str(REPO_ROOT))

DATA_DIR = REPO_ROOT / "DATA"
EPHYS_DIR = DATA_DIR / "ephys"

REGIONS = ["<REGION1>", "<REGION2>"]

WINDOW_BEFORE_S = 3.0
WINDOW_AFTER_S = 3.0

In [ ]:
from spikeinterface.core import load

sorting_data = {}

for mouse_dir in sorted(EPHYS_DIR.iterdir()):

    if not mouse_dir.is_dir():
        continue

    mouse_id = mouse_dir.name

    for region in REGIONS:

        sorting_dir = (
            mouse_dir
            / f"{region}_curated_sorting"
            / "sorting"
        )

        if not sorting_dir.exists():
            continue

        sorting = load(sorting_dir)

        sorting_data[(mouse_id, region)] = sorting

        print(
            f"{mouse_id} | {region}: "
            f"{len(sorting.unit_ids)} units"
        )

In [ ]:
from utils.ephys import prepare_exploration_epochs

epochs_data = {}

for mouse_dir in sorted(EPHYS_DIR.iterdir()):

    if not mouse_dir.is_dir():
        continue

    mouse_id = mouse_dir.name

    exploration_file = (
        mouse_dir / "exploration_segments.csv"
    )

    led_file = (
        mouse_dir / "LED_info.csv"
    )

    if not exploration_file.exists():
        continue

    if not led_file.exists():
        continue

    _, epochs_df = prepare_exploration_epochs(
        exploration_file,
        led_file,
    )

    epochs_data[mouse_id] = epochs_df

    print(
        f"{mouse_id}: "
        f"{len(epochs_df)} exploration epochs"
    )

In [ ]:
spikes_data = {}

for (mouse_id, region), sorting in sorting_data.items():

    sampling_frequency = (
        sorting.get_sampling_frequency()
    )

    spikes_data[(mouse_id, region)] = {
        unit_id: (
            sorting.get_unit_spike_train(unit_id)
            / sampling_frequency
        )
        for unit_id in sorting.unit_ids
    }

    print(
        f"{mouse_id} | {region}: "
        f"{len(spikes_data[(mouse_id, region)])} units"
    )

In [ ]:
# Prepare exploration epochs for raster alignment.
# Spike times and exploration boundaries are expressed in seconds.

for mouse_id, epochs_df in epochs_data.items():

    required_columns = {
        "label",
        "start_exploration_s",
        "end_exploration_s",
    }

    missing_columns = (
        required_columns - set(epochs_df.columns)
    )

    if missing_columns:
        raise ValueError(
            f"{mouse_id}: missing required columns "
            f"{sorted(missing_columns)}"
        )

    print(
        f"{mouse_id}: "
        f"{len(epochs_df)} epochs available"
    )

In [ ]:
# Generate and save one raster plot for each curated unit,
# aligned to the start of each exploration epoch.
# Figures are saved as SVG files in the output directory

from utils.plot import save_figure, plot_unit_raster

RASTER_DIR = (
    REPO_ROOT
    / "Fig_8"
    / "output"
    / "rasters"
)

RASTER_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

for (mouse_id, region), spikes in spikes_data.items():

    epochs = epochs_data[mouse_id]

    for unit_id in sorted(spikes.keys()):

        fig, ax = plt.subplots(
            figsize=(6, 4)
        )

        plot_unit_raster(
            spike_times=spikes[unit_id],
            epochs=epochs,
            unit_id=unit_id,
            region=region,
            mouse_id=mouse_id,
            window_before=WINDOW_BEFORE_S,
            window_after=WINDOW_AFTER_S,
            ax=ax,
        )

        filename = (
            f"{mouse_id}_{region}_"
            f"unit_{unit_id}_raster.svg"
        )

        save_figure(
            fig,
            filename,
            RASTER_DIR,
        )

        plt.close(fig)

        print(
            f"Saved: {mouse_id} | "
            f"{region} | Unit {unit_id}"
        )